[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/MecanicaCeleste/blob/main/ProyectoApophis/CarpetaDePruebas/ClaseSSB_Apophis_pymcel.ipynb)

# Clase: Posiciones, Velocidades y Dinámica del Sistema
# Sol – Tierra – Luna – Apophis respecto al SSB
## Utilizando `pymcel` y JPL Horizons

**Curso:** Mecánica Celeste  
**Institución:** Universidad de Antioquia  
**Herramientas:** `pymcel`, `scipy`, `matplotlib`, `numpy`  
**Fecha de inicio de la simulación:** 11 de abril de 2026  
**Duración de la integración:** 1 año (365 días), pasos de 1 día  
**Sistema de referencia:** Baricentro del Sistema Solar (SSB)

---

> **Referencias principales:**  
> - Zuluaga, J. I. (2024). *Mecánica Celeste: teoría, algoritmos y problemas*. Universidad de Antioquia.  
> - Repositorio `pymcel`: https://github.com/seap-udea/pymcel  
> - Repositorio `meccel-clase-profesor`: https://github.com/seap-udea/meccel-clase-profesor  
> - JPL Horizons Web Interface: https://ssd.jpl.nasa.gov/horizons/  
> - Park, R. S. et al. (2021). *AJ* 161, 105 — Efemérides DE441.


## 1. Objetivos de Aprendizaje

Al finalizar esta clase el estudiante será capaz de:

1. Explicar qué es el **Baricentro del Sistema Solar (SSB)** y por qué es el sistema de referencia inercial natural del Sistema Solar.
2. Obtener los **vectores de estado** (posición $\mathbf{r}$ y velocidad $\mathbf{v}$) de Sol, Tierra, Luna y Apophis respecto al SSB en tres dimensiones, usando el paquete `pymcel` y la base de datos JPL Horizons.
3. Interpretar el **parámetro gravitacional** $\mu_i = GM_i$ de cada cuerpo y entender su papel en las ecuaciones de movimiento.
4. Formular y resolver numéricamente el **Problema de N-Cuerpos** para el sistema Sol–Tierra–Luna–Apophis durante 1 año.
5. Verificar la conservación de energía y momento lineal como indicador de calidad numérica.
6. Producir **gráficas** 2D y 3D de las trayectorias, distancias y velocidades de cada cuerpo.


---
## 2. Marco Teórico

### 2.1 El Baricentro del Sistema Solar (SSB)

El **Baricentro del Sistema Solar** (Solar System Barycenter, SSB) es el centro de masa de todo el Sistema Solar. Si denotamos la masa del cuerpo $i$ como $m_i$ y su posición como $\mathbf{r}_i$, el SSB se define como:

$$\mathbf{R}_{\rm SSB} = \frac{\sum_i m_i \, \mathbf{r}_i}{\sum_i m_i}$$

Por definición, en el **marco del SSB** el **momento lineal total** del Sistema Solar es cero:

$$\mathbf{P}_{\rm total} = \sum_i m_i \, \mathbf{v}_i = \mathbf{0}$$

Esto lo convierte en un **sistema de referencia inercial** (o casi inercial para propósitos prácticos), lo que significa que las leyes de Newton se aplican directamente sin términos de arrastre ficticios.

| Marco de referencia | Origen | Inercial | Usado por |
|---|---|---|---|
| **SSB eclíptico J2000** | Baricentro del SS | ✅ Sí | JPL DE441, SPICE |
| Heliocéntrico eclíptico J2000 | Centro del Sol | ≈ Sí (el Sol oscila ~0.01 AU) | Astronomía clásica |
| Geocéntrico | Centro de la Tierra | ❌ No (acelerado) | Satélites, observaciones |

> **Dato curioso:** El Sol no está exactamente en el SSB. Por efecto de Júpiter y Saturno, el Sol puede estar hasta ~0.01 AU (≈1.5 millones de km, ~2 radios solares) fuera del baricentro.

### 2.2 El eje eclíptico J2000

Las posiciones y velocidades en el SSB se expresan en el sistema de coordenadas **eclíptico J2000**:
- **Eje X**: apunta hacia el equinoccio vernal del año 2000.0.
- **Eje Z**: perpendicular al plano de la eclíptica (plano de la órbita terrestre).
- **Eje Y**: completa la base derecha $\hat{z} = \hat{x} \times \hat{y}$.

Este es el sistema que usa JPL Horizons por defecto al pedir vectores de estado (`COORD_TYPE=2`).

---

### 2.3 Vectores de Estado: Posición y Velocidad

El **vector de estado** de un cuerpo en el SSB tiene 6 componentes:

$$\mathbf{y}_i = (x_i, y_i, z_i, \dot{x}_i, \dot{y}_i, \dot{z}_i) = (\mathbf{r}_i, \mathbf{v}_i)$$

donde:
- $(x_i, y_i, z_i)$ es la posición en Unidades Astronómicas (UA).
- $(\dot{x}_i, \dot{y}_i, \dot{z}_i)$ es la velocidad en UA/día.

JPL Horizons proporciona estos datos para cualquier cuerpo del Sistema Solar a partir de las **efemérides DE441** (integración numérica de alta precisión).

---

### 2.4 El Parámetro Gravitacional $\mu = GM$

La constante gravitacional universal $G = 6.674 \times 10^{-11}$ m³ kg⁻¹ s⁻² se conoce con una precisión relativa de ~22 ppm, mientras que el **parámetro gravitacional** $\mu_i = GM_i$ se conoce con muchísima mayor precisión (hasta 11 dígitos significativos) mediante observaciones orbitales de naves espaciales.

Por esta razón, en astrodinámica se trabaja directamente con $\mu$ en lugar de $G$ y $M$ por separado:

| Cuerpo | $\mu = GM$ (m³/s²) | $\mu$ (UA³/día²) | $m/M_\odot$ |
|--------|---------------------|------------------|---------------|
| **Sol** | $1.32712 \times 10^{20}$ | $2.9592 \times 10^{-4}$ | $1.0$ |
| **Tierra** | $3.9860 \times 10^{14}$ | $8.8878 \times 10^{-10}$ | $3.003 \times 10^{-6}$ |
| **Luna** | $4.9048 \times 10^{12}$ | $1.0931 \times 10^{-11}$ | $3.694 \times 10^{-8}$ |
| **Apophis** | $\approx 1.8 \times 10^{0}$ | $\approx 4 \times 10^{-24}$ | $\approx 1.4 \times 10^{-20}$ |

> **Nota:** La masa de Apophis es completamente despreciable para la dinámica del sistema. Sin embargo, su trayectoria sí es afectada por los demás cuerpos.

---

### 2.5 El Problema de N-Cuerpos: Ecuaciones de Movimiento

Para un sistema de $N$ cuerpos con masas $m_i$ y posiciones $\mathbf{r}_i$ respecto al SSB, las ecuaciones de movimiento de Newton son:

$$\boxed{\ddot{\mathbf{r}}_i = \sum_{j \neq i} \frac{G m_j (\mathbf{r}_j - \mathbf{r}_i)}{|\mathbf{r}_j - \mathbf{r}_i|^3}}$$

En nuestra simulación tenemos $N = 4$ cuerpos: Sol, Tierra, Luna y Apophis. Esto nos da un sistema de $4 \times 3 = 12$ EDOs de segundo orden, o equivalentemente $24$ EDOs de primer orden.

**Constantes de movimiento:**
- Energía mecánica total: $E = \frac{1}{2}\sum_i m_i v_i^2 - \sum_{i<j} \frac{G m_i m_j}{|\mathbf{r}_i - \mathbf{r}_j|}$
- Momento lineal total: $\mathbf{P} = \sum_i m_i \mathbf{v}_i = \mathbf{0}$ (en el SSB)
- Momento angular total: $\mathbf{L} = \sum_i m_i (\mathbf{r}_i \times \mathbf{v}_i)$

---

### 2.6 Sistema de Unidades

Para esta simulación usaremos el sistema de unidades astronómicas-día:

| Cantidad | Unidad | Símbolo |
|----------|--------|---------|
| Longitud | Unidad Astronómica | UA |
| Tiempo | día | d |
| Masa (como $\mu$) | UA³/d² | — |

En este sistema, la constante gravitacional se absorbe en la masa: $m_i^{(\text{sim})} \equiv G m_i = \mu_i$ expresado en UA³/d².

La velocidad orbital circular de la Tierra a 1 UA es:
$$v_\oplus = \sqrt{\mu_\odot / 1\, \text{UA}} = \sqrt{2.9592 \times 10^{-4}} \approx 0.01721\, \text{UA/d} \approx 29.8\, \text{km/s}$$

Horizons devuelve posiciones en UA y velocidades en UA/día, así que **no se necesita ninguna conversión** para las condiciones iniciales.


---
## 3. Instalación e Importaciones

Instalamos `pymcel` y los paquetes necesarios:


In [1]:
!pip install "numpy<2.3"

In [2]:
# Instalación de dependencias
# Ejecutar esta celda la primera vez (en Colab o entorno limpio)
!pip install pymcel astroquery astropy scipy matplotlib -Uq

print('✓ Paquetes instalados correctamente')


✓ Paquetes instalados correctamente


In [4]:
# ─── Importaciones principales ──────────────────────────────────────────
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# pymcel: paquete de Mecánica Celeste (Zuluaga, UdeA)
import pymcel as pc

# Constantes del paquete pymcel
try:
    from pymcel.constantes import GM_sun, GM_earth, GM_jup, au, year
    from pymcel.constantes import mu_sun, mu_earth, mu_moon
    print('✓ Constantes pymcel cargadas correctamente')
except ImportError:
    print('Cargando constantes manualmente...')
    # Constantes de respaldo (valores DE441 / IAU 2015)
    GM_sun   = 1.32712440018e20   # m^3 s^-2
    GM_earth = 3.986004418e14     # m^3 s^-2
    mu_moon  = 4.90486959e12      # m^3 s^-2

# scipy para integración y álgebra lineal
from scipy.integrate import solve_ivp
from scipy.linalg import norm

print(f'\n  numpy  : {np.__version__}')
print(f'  matplotlib: {matplotlib.__version__}')

# Estilo de gráficas
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'font.size': 11,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
})
print('\n✓ Importaciones completadas')


Cargando constantes manualmente...

  numpy  : 2.2.6
  matplotlib: 3.10.8

✓ Importaciones completadas


---
## 4. Constantes Físicas y Parámetros Gravitacionales ($\mu = GM$)

Definimos todas las constantes necesarias en el sistema de unidades UA–día.

**Conversión de $\mu$ de SI (m³/s²) a UA³/día²:**
$$\mu_{\rm UA^3/d^2} = \mu_{\rm m^3/s^2} \cdot \frac{(86400)^2}{(1.495978707 \times 10^{11})^3}$$


In [ ]:
# ─── Constantes de conversión ────────────────────────────────────────────
AU_m    = 1.495978707e11    # 1 UA en metros
day_s   = 86400.0           # 1 día en segundos
yr_d    = 365.25            # 1 año en días
G_SI    = 6.67430e-11       # Constante gravitacional [m^3 kg^-1 s^-2]
c_luz   = 2.99792458e8      # Velocidad de la luz [m/s]

# Factor de conversión de mu: m^3/s^2 → UA^3/día^2
fac_mu = day_s**2 / AU_m**3
print(f'Factor de conversión μ (m³/s² → UA³/d²): {fac_mu:.6e}')

# ─── Parámetros gravitacionales μ = GM [m³/s²] ──────────────────────────
# Fuente: IAU 2015 / Horizons DE441
mu_sun_SI     = 1.32712440018e20   # m^3/s^2
mu_earth_SI   = 3.986004418e14     # m^3/s^2  (solo Tierra, sin Luna)
mu_moon_SI    = 4.90486959e12      # m^3/s^2
mu_apophis_SI = G_SI * 2.7e10      # m^3/s^2  (masa ~2.7×10^10 kg)

# ─── Conversión a UA³/día² ───────────────────────────────────────────────
MU_SOL     = mu_sun_SI   * fac_mu
MU_TIERRA  = mu_earth_SI * fac_mu
MU_LUNA    = mu_moon_SI  * fac_mu
MU_APOPHIS = mu_apophis_SI * fac_mu

# ─── Tabla de masas y μ ─────────────────────────────────────────────────
print('\n' + '='*70)
print(f'{"Cuerpo":<12} {"μ (m³/s²)":>18} {"μ (UA³/d²)":>16} {"m/M_sol":>14}')
print('-'*70)
cuerpos_mu = [
    ('Sol',     mu_sun_SI,     MU_SOL,     1.0),
    ('Tierra',  mu_earth_SI,   MU_TIERRA,  mu_earth_SI / mu_sun_SI),
    ('Luna',    mu_moon_SI,    MU_LUNA,    mu_moon_SI  / mu_sun_SI),
    ('Apophis', mu_apophis_SI, MU_APOPHIS, mu_apophis_SI / mu_sun_SI),
]
for nombre, mu_si, mu_ua, ratio in cuerpos_mu:
    print(f'{nombre:<12} {mu_si:>18.6e} {mu_ua:>16.6e} {ratio:>14.4e}')
print('='*70)

# Verificación: velocidad orbital circular de la Tierra
v_tierra_circ = np.sqrt(MU_SOL / 1.0)  # UA/día a 1 UA
v_tierra_km_s = v_tierra_circ * AU_m / day_s / 1e3
print(f'\n  Vel. circular Tierra a 1 UA: {v_tierra_circ:.5f} UA/d = {v_tierra_km_s:.2f} km/s')
print(f'  (Valor real: ~29.78 km/s — error: {abs(v_tierra_km_s-29.78)/29.78*100:.2f} %)')


---
## 5. Condiciones Iniciales desde JPL Horizons

### 5.1 ¿Qué es JPL Horizons?

**JPL Horizons** (https://ssd.jpl.nasa.gov/horizons/) es el sistema de efemérides de alta precisión del Jet Propulsion Laboratory de la NASA. Proporciona posiciones y velocidades de cualquier objeto del Sistema Solar respecto a cualquier sistema de referencia.

Usamos la integración numérica **DE441** que incluye las perturbaciones de todos los planetas, la Luna y los asteroides más masivos.

### 5.2 Tabla de Identificadores

| Cuerpo | ID Horizons | Código en la simulación |
|--------|-------------|------------------------|
| Sol | `10` | `sol` |
| Tierra | `399` | `tierra` |
| Luna | `301` | `luna` |
| Apophis | `99942` | `apophis` |

El parámetro `location='@0'` indica que el **origen es el SSB** (código 0 en Horizons).

### 5.3 Variables de tipo `vectors` en Horizons

Al solicitar `TABLE_TYPE=VECTORS`, Horizons devuelve:
- $x, y, z$: posición en UA respecto al SSB
- $v_x, v_y, v_z$: velocidad en UA/día respecto al SSB
- $\Delta t$: tiempo desde la época en días

> **Nota sobre pymcel:** La función `pc.consulta_horizons()` envuelve la API de `astroquery.jplhorizons.Horizons` y devuelve un `pandas.DataFrame` con las columnas anteriores. Para cuerpos pequeños como Apophis usamos `id_type='smallbody'`.


In [ ]:
# ─── Época de inicio de la simulación ───────────────────────────────────
EPOCH = '2026-Apr-11'   # 11 de abril de 2026

# Configuración de la consulta a Horizons
# Pedimos solo el instante inicial (start=stop)
epochs_ini = {'start': '2026-04-11', 'stop': '2026-04-12', 'step': '1d'}

# ─── Lista de cuerpos: (nombre, ID Horizons, tipo, mu en UA³/d²) ─────────
CUERPOS = [
    ('Sol',     '10',    'majorbody', MU_SOL),
    ('Tierra',  '399',   'majorbody', MU_TIERRA),
    ('Luna',    '301',   'majorbody', MU_LUNA),
    ('Apophis', '99942', 'smallbody', MU_APOPHIS),
]

# Colores para cada cuerpo (para gráficas)
COLORES = {
    'Sol':     '#FFA500',  # naranja
    'Tierra':  '#1f77b4',  # azul
    'Luna':    '#888888',  # gris
    'Apophis': '#d62728',  # rojo
}

print(f'Época inicial de la simulación: {EPOCH}')
print(f'Cuerpos incluidos en la simulación: {[c[0] for c in CUERPOS]}')


### 5.4 Obtención de los vectores de estado

Llamamos a `pc.consulta_horizons()` para cada cuerpo.

**¿Cómo interpreta Horizons los IDs?**
- Para planetas y lunas: ID numérico (`'399'`, `'301'`, `'10'`).
- Para asteroides y cuerpos pequeños: ID del SBDB sin punto y coma, o nombre (`'99942'`).

La función devuelve un `DataFrame` de pandas. Tomamos la **primera fila** (época = 11 abril 2026).


In [ ]:
# ─── Obtención de condiciones iniciales via pymcel + Horizons ────────────

estados_ini = {}   # Diccionario: nombre -> {'r': [x,y,z], 'v': [vx,vy,vz]}

for nombre, horizons_id, id_type, mu in CUERPOS:
    print(f'Consultando Horizons para {nombre} (ID={horizons_id})...')
    try:
        # Intento con pymcel
        df = pc.consulta_horizons(
            id=horizons_id,
            location='@0',          # SSB como origen
            epochs=epochs_ini,
            datos='vectors',
        )
        fila = df.iloc[0]
        r = np.array([float(fila['x']),  float(fila['y']),  float(fila['z'])])
        v = np.array([float(fila['vx']), float(fila['vy']), float(fila['vz'])])
    except Exception as e:
        print(f'  [pymcel falló: {e}] — usando astroquery directamente...')
        from astroquery.jplhorizons import Horizons
        obj = Horizons(
            id=horizons_id,
            location='@0',
            epochs=epochs_ini,
            id_type=id_type if id_type != 'majorbody' else None,
        )
        vecs = obj.vectors()
        fila = vecs[0]
        r = np.array([fila['x'],  fila['y'],  fila['z']])
        v = np.array([fila['vx'], fila['vy'], fila['vz']])

    estados_ini[nombre] = {'r': r, 'v': v, 'mu': mu}
    dist_ssb = np.linalg.norm(r)
    vel_mag  = np.linalg.norm(v)
    print(f'  r_SSB = {dist_ssb:.4f} UA | v = {vel_mag:.6f} UA/d')

print('\n✓ Condiciones iniciales obtenidas para todos los cuerpos')


### 5.5 Resumen del vector de estado inicial

Mostramos la tabla completa del estado en la época inicial (11 de abril de 2026):


In [ ]:
# ─── Tabla del estado inicial en el SSB ─────────────────────────────────
print('\n' + '='*90)
print(f'ESTADO INICIAL — Época: {EPOCH}  |  Marco: SSB eclíptico J2000')
print('='*90)
header = f'{"Cuerpo":<10} {"x (UA)":>12} {"y (UA)":>12} {"z (UA)":>12}  '\
         f'{"vx (UA/d)":>12} {"vy (UA/d)":>12} {"vz (UA/d)":>12}'
print(header)
print('-'*90)
for nombre, data in estados_ini.items():
    r, v = data['r'], data['v']
    print(f'{nombre:<10} {r[0]:>12.6f} {r[1]:>12.6f} {r[2]:>12.6f}  '
          f'{v[0]:>12.8f} {v[1]:>12.8f} {v[2]:>12.8f}')
print('='*90)

# ─── Verificación: momento lineal total ≈ 0 ──────────────────────────────
P_total = np.zeros(3)
M_total = 0.0
for nombre, data in estados_ini.items():
    mi = data['mu']   # μ_i ≡ G*m_i  (unidades masa canónicas)
    P_total += mi * data['v']
    M_total += mi

# En el SSB, P_total debería ser ≈ 0
# (el Sol domina la masa, así que es un buen diagnóstico)
print(f'\nMomento lineal total (en unidades μ·v):')
print(f'  Px = {P_total[0]:.3e}  Py = {P_total[1]:.3e}  Pz = {P_total[2]:.3e}')
print('  (Valores ≈ 0 confirman que el SSB es nuestro origen inercial)')


### 5.6 Visualización del estado inicial (escena 3D)

Visualizamos las posiciones iniciales de los cuatro cuerpos en el espacio 3D respecto al SSB.


In [ ]:
# ─── Gráfica 3D del estado inicial ──────────────────────────────────────
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Tamaños de los marcadores
tam = {'Sol': 200, 'Tierra': 60, 'Luna': 30, 'Apophis': 20}

for nombre, data in estados_ini.items():
    r = data['r']
    color = COLORES[nombre]
    ax.scatter(*r, s=tam[nombre], color=color, label=nombre, zorder=5, edgecolors='k', linewidths=0.4)
    # Vectores de velocidad (escalados para visibilidad)
    v = data['v']
    escala = 15.0  # días de velocidad para visualizar
    ax.quiver(r[0], r[1], r[2], v[0]*escala, v[1]*escala, v[2]*escala,
              color=color, alpha=0.7, linewidth=1.2)
    ax.text(r[0], r[1], r[2]+0.03, nombre, fontsize=9, color=color)

# Marcar el SSB
ax.scatter(0, 0, 0, s=40, color='k', marker='+', zorder=6)
ax.text(0, 0, 0.05, 'SSB', fontsize=8, color='k')

ax.set_xlabel('x (UA)')
ax.set_ylabel('y (UA)')
ax.set_zlabel('z (UA)')
ax.set_title(f'Estado inicial — {EPOCH} — Marco SSB eclíptico J2000', fontsize=12)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()
print('Las flechas indican la dirección de la velocidad (escalada × 15 días).')


---
## 6. Configuración y Ejecución de la Simulación de N-Cuerpos

### 6.1 Formato del sistema para `pymcel`

El paquete `pymcel` provee la función `pc.ncuerpos_solucion(sistema, ts)` que integra numéricamente las ecuaciones de movimiento del Problema de N-Cuerpos usando `scipy.integrate.odeint` con paso variable.

El parámetro `sistema` es una lista de diccionarios, uno por cuerpo:
```python
sistema = [
    dict(m=mu_i,  # parámetro gravitacional μ_i = GM_i [UA³/d²] (G=1 canónico)
         r=[x, y, z],       # posición inicial [UA]
         v=[vx, vy, vz]),   # velocidad inicial [UA/d]
    ...
]
```

El vector de tiempos `ts` está en días (unidad canónica de tiempo elegida).

> **¿Por qué G=1?** En las unidades UA–día, hacemos $m_i^{(\rm sim)} = \mu_i = GM_i$. Entonces la ecuación de movimiento $\ddot{r} = G \cdot m_j (r_j - r_i)/|...|^3$ se convierte en $\ddot{r} = m_j^{(\rm sim)} (r_j - r_i)/|...|^3$, con G implícitamente igual a 1.

### 6.2 Parámetros de la simulación

| Parámetro | Valor |
|-----------|-------|
| Fecha inicial | 11 de abril de 2026 |
| Duración | 365 días (1 año) |
| Paso de salida | 1 día |
| N° de pasos | 366 (t = 0, 1, 2, ..., 365) |
| Integrador | `scipy.integrate.odeint` (Runge-Kutta adaptativo, `pymcel`) |
| Sistema de referencia | SSB eclíptico J2000 |


In [ ]:
# ─── Construcción del sistema para pymcel ────────────────────────────────

sistema = []
nombres_sim = []  # para etiquetas

for nombre, _, _, mu in CUERPOS:
    data = estados_ini[nombre]
    sistema.append(dict(
        m = mu,          # μ = GM en UA^3/d^2 (G=1 canónico)
        r = list(data['r']),
        v = list(data['v']),
    ))
    nombres_sim.append(nombre)

# Imprimir configuración
print('Sistema de simulación:')
print(f'{"Cuerpo":<10} {"μ (UA³/d²)":>16} {"r [UA]":<40} {"v [UA/d]"}')
print('-'*100)
for i, (cuerpo, s) in enumerate(zip(nombres_sim, sistema)):
    rv = np.array(s['r'])
    vv = np.array(s['v'])
    print(f'{cuerpo:<10} {s["m"]:>16.4e}  [{rv[0]:+.4f}, {rv[1]:+.4f}, {rv[2]:+.5f}]  '
          f'[{vv[0]:+.6f}, {vv[1]:+.6f}, {vv[2]:+.6f}]')


In [ ]:
# ─── Vector de tiempos: 0 a 365 días, paso 1 día ─────────────────────────
N_PASOS = 366
T_FINAL = 365.0    # días
ts = np.linspace(0.0, T_FINAL, N_PASOS)

print(f'Vector de tiempos:')
print(f'  t_inicial = {ts[0]:.1f} d  (11 abr 2026)')
print(f'  t_final   = {ts[-1]:.1f} d (= t + 365 días → 11 abr 2027)')
print(f'  N° de pasos de salida: {len(ts)}')
print(f'  Paso: 1 día')


### 6.3 Ejecución de la integración

Llamamos a `pc.ncuerpos_solucion(sistema, ts)` que devuelve:
- `rs`: posiciones $\mathbf{r}_i(t)$ para cada cuerpo, en UA. Forma: `(N_cuerpos, N_pasos, 3)`
- `vs`: velocidades $\mathbf{v}_i(t)$ para cada cuerpo, en UA/día. Forma: `(N_cuerpos, N_pasos, 3)`
- `rps`: posiciones relativas al centro de masa.
- `vps`: velocidades relativas al centro de masa.
- `constantes`: diccionario con energía total $E(t)$ y momento angular $\mathbf{L}(t)$.


In [ ]:
# ─── Integración de N-Cuerpos con pymcel ─────────────────────────────────
import time

print('Iniciando integración de N-Cuerpos...')
print(f'  Cuerpos: {nombres_sim}')
print(f'  Duración: {T_FINAL:.0f} días con {N_PASOS} pasos de salida')

t_inicio = time.time()
rs, vs, rps, vps, constantes = pc.ncuerpos_solucion(sistema, ts)
t_fin = time.time()

print(f'\n✓ Integración completada en {t_fin - t_inicio:.2f} segundos')
print(f'  Forma de rs: {rs.shape}  → (N_cuerpos={rs.shape[0]}, N_pasos={rs.shape[1]}, xyz=3)')
print(f'  Forma de vs: {vs.shape}')

# Índices para acceder a cada cuerpo
I_SOL     = 0
I_TIERRA  = 1
I_LUNA    = 2
I_APOPHIS = 3

# Extraer trayectorias individuales
r_sol     = rs[I_SOL]       # shape (N_pasos, 3)
r_tierra  = rs[I_TIERRA]
r_luna    = rs[I_LUNA]
r_apophis = rs[I_APOPHIS]

v_sol     = vs[I_SOL]
v_tierra  = vs[I_TIERRA]
v_luna    = vs[I_LUNA]
v_apophis = vs[I_APOPHIS]

print(f'\nPosición final del Sol: ({r_sol[-1,0]:.4f}, {r_sol[-1,1]:.4f}, {r_sol[-1,2]:.4f}) UA')
print(f'Posición final de Tierra: ({r_tierra[-1,0]:.4f}, {r_tierra[-1,1]:.4f}, {r_tierra[-1,2]:.4f}) UA')


---
## 7. Validación: Conservación de Energía y Momento Angular

Una forma de verificar la calidad de la integración numérica es comprobar que las **constantes de movimiento** se conserven bien. Para el Problema de N-Cuerpos, las constantes son:

1. **Energía total** $E = T + V$ (cinética + potencial): debe conservarse dentro de la precisión numérica.
2. **Momento angular total** $\mathbf{L} = \sum_i m_i (\mathbf{r}_i \times \mathbf{v}_i)$: constante en módulo y dirección.

El error relativo en la energía nos da una medida directa de la calidad del integrador:
$$\epsilon_E = \left| \frac{E(t) - E(0)}{E(0)} \right|$$


In [ ]:
# ─── Verificación de conservación de energía ─────────────────────────────

mus = np.array([s['m'] for s in sistema])  # μ_i para cada cuerpo

def energia_total(rs_t, vs_t, mus):
    """
    Calcula la energía total del sistema de N cuerpos.
    rs_t: array (N, 3) de posiciones en un instante t
    vs_t: array (N, 3) de velocidades
    mus:  array (N,) de mu_i = G*m_i
    Retorna la energía en unidades de μ·UA/d²
    """
    N = len(mus)
    # Energía cinética
    T = 0.5 * np.sum(mus * np.sum(vs_t**2, axis=1))
    # Energía potencial (negativa)
    V = 0.0
    for i in range(N):
        for j in range(i+1, N):
            r_ij = np.linalg.norm(rs_t[i] - rs_t[j])
            V -= mus[i] * mus[j] / r_ij  # V = -G*m_i*m_j/r (G=1)
    return T + V

def momento_angular(rs_t, vs_t, mus):
    """
    Momento angular total del sistema.
    """
    L = np.zeros(3)
    for i in range(len(mus)):
        L += mus[i] * np.cross(rs_t[i], vs_t[i])
    return L

# Calcular E(t) y |L(t)| para todos los pasos
print('Calculando energía y momento angular en cada paso...')
E_t = np.array([
    energia_total(rs[:, k, :], vs[:, k, :], mus) for k in range(N_PASOS)
])
L_t = np.array([
    np.linalg.norm(momento_angular(rs[:, k, :], vs[:, k, :], mus)) for k in range(N_PASOS)
])

# Error relativo
eps_E = np.abs((E_t - E_t[0]) / E_t[0])
eps_L = np.abs((L_t - L_t[0]) / L_t[0])

print(f'\n  Energía inicial: E₀ = {E_t[0]:.6e}')
print(f'  Error máximo en E: {eps_E.max():.2e}')
print(f'  Error max en |L|:  {eps_L.max():.2e}')

if eps_E.max() < 1e-6:
    print('  ✓ Conservación excelente (error < 1e-6)')
elif eps_E.max() < 1e-4:
    print('  ⚠ Conservación aceptable (error < 1e-4)')
else:
    print('  ✗ Error grande — considerar paso más pequeño')


In [ ]:
# ─── Gráfica de conservación ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

ax = axes[0]
ax.semilogy(ts, eps_E, 'b-', linewidth=1.5, label=r'$|\Delta E / E_0|$')
ax.set_ylabel('Error relativo en energía')
ax.set_title('Conservación de las constantes de movimiento')
ax.legend()
ax.set_ylim(bottom=1e-16)

ax = axes[1]
ax.semilogy(ts, eps_L + 1e-20, 'r-', linewidth=1.5, label=r'$|\Delta L / L_0|$')
ax.set_xlabel('Tiempo (días desde 11 abr 2026)')
ax.set_ylabel('Error relativo en |L|')
ax.legend()

plt.tight_layout()
plt.show()
print('Si los errores son menores a 1e-6, la integración es confiable.')


---
## 8. Visualización de las Trayectorias

### 8.1 Trayectorias en 3D respecto al SSB

Graficamos la posición de cada cuerpo durante el año de simulación. El Sol se mueve muy poco respecto al SSB (el SSB está muy cerca del Sol), mientras que la Tierra y Apophis trazan sus órbitas alrededor de él.


In [ ]:
# ─── Trayectoria 3D de los 4 cuerpos ─────────────────────────────────────
fig = plt.figure(figsize=(13, 10))
ax = fig.add_subplot(111, projection='3d')

trajs = {
    'Sol':     r_sol,
    'Tierra':  r_tierra,
    'Luna':    r_luna,
    'Apophis': r_apophis,
}
lw = {'Sol': 1.5, 'Tierra': 1.5, 'Luna': 0.8, 'Apophis': 1.2}
sz = {'Sol': 150, 'Tierra': 60, 'Luna': 25, 'Apophis': 20}

for nombre, tray in trajs.items():
    c = COLORES[nombre]
    ax.plot(tray[:,0], tray[:,1], tray[:,2], '-', color=c,
            linewidth=lw[nombre], label=nombre, alpha=0.85)
    # Punto inicial
    ax.scatter(*tray[0], s=sz[nombre], color=c, edgecolors='k', linewidths=0.5, zorder=5)
    # Punto final (estrella)
    ax.scatter(*tray[-1], s=sz[nombre], color=c, marker='*', zorder=5)

# SSB
ax.scatter(0, 0, 0, s=40, color='k', marker='+', zorder=10)
ax.text(0, 0, 0.05, 'SSB', fontsize=8)

ax.set_xlabel('x (UA)')
ax.set_ylabel('y (UA)')
ax.set_zlabel('z (UA)')
ax.set_title(
    'Trayectorias 3D — Sol, Tierra, Luna y Apophis respecto al SSB\n'
    '11 abr 2026 → 11 abr 2027  (•=inicio, ★=fin)',
    fontsize=11
)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()


### 8.2 Trayectorias en los planos coordenados (XY, XZ, YZ)

Las proyecciones en los tres planos coordenados nos permiten ver con más claridad la inclinación de las órbitas y la diferencia en escala entre los cuerpos.


In [ ]:
# ─── Proyecciones en planos coordenados ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

planos = [
    (0, 1, 'x (UA)', 'y (UA)', 'Plano XY (eclíptica ≈)'),
    (0, 2, 'x (UA)', 'z (UA)', 'Plano XZ'),
    (1, 2, 'y (UA)', 'z (UA)', 'Plano YZ'),
]

for ax, (ix, iy, xl, yl, titulo) in zip(axes, planos):
    for nombre, tray in trajs.items():
        c = COLORES[nombre]
        ax.plot(tray[:,ix], tray[:,iy], '-', color=c,
                linewidth=lw[nombre], label=nombre, alpha=0.85)
        ax.scatter(tray[0,ix], tray[0,iy], s=40, color=c,
                   edgecolors='k', linewidths=0.4, zorder=5)

    ax.scatter(0, 0, s=25, color='k', marker='+', zorder=6)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.set_title(titulo)
    ax.set_aspect('equal')
    if ax == axes[0]:
        ax.legend(fontsize=8)

plt.suptitle('Proyecciones de las trayectorias — Marco SSB eclíptico J2000', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('Nota: el plano XY se aproxima al plano de la eclíptica (órbita de la Tierra).')
print('La inclinación de Apophis (i ≈ 3.3°) es visible en los planos XZ e YZ.')


### 8.3 Vista ampliada: Sistema Tierra–Luna–Apophis

Ampliamos la región cercana a la Tierra para ver la órbita de la Luna y la trayectoria de Apophis **en coordenadas relativas a la Tierra** (marco geocéntrico).


In [ ]:
# ─── Coordenadas geocéntricas ─────────────────────────────────────────────
# Posiciones relativas a la Tierra
r_luna_geo    = r_luna    - r_tierra   # Luna respecto a Tierra
r_apophis_geo = r_apophis - r_tierra   # Apophis respecto a Tierra

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Plano XY geocéntrico ────
ax = axes[0]
ax.plot(r_luna_geo[:,0], r_luna_geo[:,1], '-', color=COLORES['Luna'],
        linewidth=0.8, label='Luna', alpha=0.85)
ax.plot(r_apophis_geo[:,0], r_apophis_geo[:,1], '-', color=COLORES['Apophis'],
        linewidth=1.0, label='Apophis', alpha=0.90)
ax.scatter(0, 0, s=80, color=COLORES['Tierra'], label='Tierra', zorder=6, edgecolors='k')
ax.set_xlabel('Δx (UA)')
ax.set_ylabel('Δy (UA)')
ax.set_title('Marco geocéntrico — Plano XY')
ax.legend()
ax.set_aspect('equal')

# ── Distancia Tierra-Apophis vs tiempo ────
dist_TiA = np.linalg.norm(r_apophis_geo, axis=1)  # en UA
dist_TiA_km = dist_TiA * AU_m / 1e3
dist_TiA_LD = dist_TiA * AU_m / 3.844e8  # distancias lunares

ax2 = axes[1]
ax2_twin = ax2.twinx()
l1, = ax2.plot(ts, dist_TiA, color=COLORES['Apophis'], linewidth=1.5, label='Distancia T-A (UA)')
l2, = ax2_twin.plot(ts, dist_TiA_LD, color='purple', linewidth=1.0, linestyle='--',
                    label='Distancia T-A (LD)', alpha=0.7)

idx_min = np.argmin(dist_TiA)
ax2.axvline(ts[idx_min], color='gray', linestyle=':', alpha=0.7)
ax2.scatter(ts[idx_min], dist_TiA[idx_min], s=60, color='red', zorder=5)
ax2.annotate(f't={ts[idx_min]:.0f}d\nd={dist_TiA[idx_min]:.4f}UA',
             xy=(ts[idx_min], dist_TiA[idx_min]),
             xytext=(ts[idx_min]+15, dist_TiA[idx_min]+0.05),
             arrowprops=dict(arrowstyle='->', color='k'), fontsize=8)

ax2.set_xlabel('Tiempo (días desde 11 abr 2026)')
ax2.set_ylabel('Distancia Tierra–Apophis (UA)', color=COLORES['Apophis'])
ax2_twin.set_ylabel('Distancia (LD = dist. lunar)', color='purple')
ax2.set_title('Distancia Tierra–Apophis durante la simulación')
ax2.legend(handles=[l1, l2], loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print(f'Máxima aproximación Tierra–Apophis en la simulación:')
print(f'  t = {ts[idx_min]:.1f} días ({ts[idx_min]/365.25:.2f} años desde inicio)')
print(f'  d = {dist_TiA[idx_min]:.6f} UA = {dist_TiA_km[idx_min]:.0f} km = {dist_TiA_LD[idx_min]:.2f} LD')


---
## 9. Distancias desde el SSB vs. Tiempo

La distancia de un cuerpo al SSB es la norma de su vector de posición:
$$d_i(t) = |\mathbf{r}_i(t)| = \sqrt{x_i^2 + y_i^2 + z_i^2}$$

Para el Sol, esta distancia refleja la **perturbación de los planetas** sobre el SSB. Para la Tierra y Apophis, refleja la forma de sus órbitas (elipses) alrededor del Sol.


In [ ]:
# ─── Distancias al SSB ────────────────────────────────────────────────────
d_sol     = np.linalg.norm(r_sol,     axis=1)
d_tierra  = np.linalg.norm(r_tierra,  axis=1)
d_luna    = np.linalg.norm(r_luna,    axis=1)
d_apophis = np.linalg.norm(r_apophis, axis=1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

datos_d = [
    ('Sol',     d_sol,     axes[0,0]),
    ('Tierra',  d_tierra,  axes[0,1]),
    ('Luna',    d_luna,    axes[1,0]),
    ('Apophis', d_apophis, axes[1,1]),
]

for nombre, d, ax in datos_d:
    c = COLORES[nombre]
    ax.plot(ts, d, '-', color=c, linewidth=1.5)
    ax.axhline(d.mean(), color=c, linestyle='--', alpha=0.5,
               label=f'Media = {d.mean():.4f} UA')
    ax.fill_between(ts, d.min(), d, alpha=0.08, color=c)
    ax.set_ylabel('Distancia al SSB (UA)')
    ax.set_title(f'{nombre}: d_SSB(t)')
    ax.legend(fontsize=9)
    # Anotaciones de mín/máx
    ax.annotate(f'min={d.min():.4f}', xy=(ts[d.argmin()], d.min()),
                xytext=(ts[d.argmin()]+10, d.min()+0.002),
                fontsize=7, color='k',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))
    ax.annotate(f'max={d.max():.4f}', xy=(ts[d.argmax()], d.max()),
                xytext=(ts[d.argmax()]-60, d.max()-0.005),
                fontsize=7, color='k',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

for ax in axes[1]:
    ax.set_xlabel('Tiempo (días desde 11 abr 2026)')

plt.suptitle('Distancias al SSB vs. Tiempo — Simulación de 1 año', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Tabla resumen
print('\nResumen estadístico de distancias al SSB (UA):')
print(f'{"Cuerpo":<10} {"Min":>10} {"Max":>10} {"Media":>10} {"Rango":>10}')
print('-'*50)
for nombre, d, _ in datos_d:
    print(f'{nombre:<10} {d.min():>10.5f} {d.max():>10.5f} {d.mean():>10.5f} {d.max()-d.min():>10.5f}')


### 9.1 Distancia Tierra–Luna

La distancia Tierra–Luna varía entre ~356,000 km (perigeo) y ~407,000 km (apogeo) con un período orbital de ~27.3 días. Este patrón oscilatorio debe ser visible claramente en la simulación.


In [ ]:
# ─── Distancia Tierra–Luna ────────────────────────────────────────────────
d_TL = np.linalg.norm(r_luna - r_tierra, axis=1)  # UA
d_TL_km = d_TL * AU_m / 1e3

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Distancia en km
ax = axes[0]
ax.plot(ts, d_TL_km, color=COLORES['Luna'], linewidth=1.2)
ax.axhline(d_TL_km.mean(), color='gray', linestyle='--', alpha=0.6,
           label=f'Media = {d_TL_km.mean():.0f} km')
ax.axhline(356500, color='b', linestyle=':', alpha=0.4, label='Perigeo medio ~356,500 km')
ax.axhline(406700, color='r', linestyle=':', alpha=0.4, label='Apogeo medio ~406,700 km')
ax.set_ylabel('Distancia Tierra–Luna (km)')
ax.set_title('Variación de la distancia Tierra–Luna durante 1 año de simulación')
ax.legend(fontsize=9)

# Período estimado por FFT
from numpy.fft import fft, fftfreq
N = len(d_TL)
freqs = fftfreq(N, d=1.0)  # frecuencias en 1/día
potencia = np.abs(fft(d_TL - d_TL.mean()))**2
# Solo frecuencias positivas
idx_pos = freqs > 0
freq_pico = freqs[idx_pos][np.argmax(potencia[idx_pos])]
T_orbital_luna = 1.0 / freq_pico  # días

# Variación de la distancia respecto a la media (para ver el patrón)
ax2 = axes[1]
ax2.plot(ts, d_TL_km - d_TL_km.mean(), color=COLORES['Luna'], linewidth=1.2)
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Tiempo (días desde 11 abr 2026)')
ax2.set_ylabel('Δ distancia T-L (km)')
ax2.set_title(f'Variación respecto a la media — Período orbital estimado: {T_orbital_luna:.1f} días')

plt.tight_layout()
plt.show()

print(f'Período orbital de la Luna estimado por FFT: {T_orbital_luna:.2f} días')
print(f'  (Valor esperado: ~27.3 días — período sidéreo de la Luna)')
print(f'Distancia mínima (perigeo): {d_TL_km.min():.0f} km')
print(f'Distancia máxima (apogeo):  {d_TL_km.max():.0f} km')


---
## 10. Módulo de la Velocidad vs. Tiempo

El módulo de la velocidad de cada cuerpo refleja la conservación de la energía de su órbita. En una órbita kepleriana elíptica, la velocidad es mayor en el **perihelio** y menor en el **afelio** (Ley de las áreas de Kepler).

Por la Ley Vis-viva:
$$v^2 = GM_\odot\left(\frac{2}{r} - \frac{1}{a}\right)$$

donde $a$ es el semieje mayor de la órbita.


In [ ]:
# ─── Velocidades de cada cuerpo ──────────────────────────────────────────
mag_v = {
    'Sol':     np.linalg.norm(v_sol,     axis=1) * AU_m / day_s / 1e3,  # km/s
    'Tierra':  np.linalg.norm(v_tierra,  axis=1) * AU_m / day_s / 1e3,
    'Luna':    np.linalg.norm(v_luna,    axis=1) * AU_m / day_s / 1e3,
    'Apophis': np.linalg.norm(v_apophis, axis=1) * AU_m / day_s / 1e3,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
axes = axes.flatten()

for i, (nombre, v_mag) in enumerate(mag_v.items()):
    ax = axes[i]
    c = COLORES[nombre]
    ax.plot(ts, v_mag, '-', color=c, linewidth=1.5)
    ax.fill_between(ts, v_mag.min(), v_mag, alpha=0.08, color=c)
    ax.axhline(v_mag.mean(), color=c, linestyle='--', alpha=0.5,
               label=f'Media = {v_mag.mean():.2f} km/s')
    ax.set_ylabel('Velocidad (km/s)')
    ax.set_title(f'{nombre}: |v(t)|')
    ax.legend(fontsize=9)
    if i >= 2:
        ax.set_xlabel('Tiempo (días desde 11 abr 2026)')

plt.suptitle('Módulo de la velocidad vs. Tiempo — SSB eclíptico J2000', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Velocidades medias:')
for nombre, v_mag in mag_v.items():
    print(f'  {nombre:<10}: {v_mag.mean():.3f} km/s  '
          f'  (min={v_mag.min():.3f}, max={v_mag.max():.3f} km/s)')


In [ ]:
# ─── Vis-viva: verificación de la ley de Kepler ──────────────────────────
print('Verificación de la Ley Vis-viva para la Tierra:')
print('  v² = μ_sol (2/r - 1/a)  → calculamos a a partir de la energía')
print()

# Vector posición y velocidad Tierra relativo al Sol
r_T_S = r_tierra - r_sol   # posición relativa al Sol (UA)
v_T_S = v_tierra - v_sol   # velocidad relativa al Sol (UA/d)

d_T_S = np.linalg.norm(r_T_S, axis=1)  # distancia Tierra-Sol
v_T_S_mag = np.linalg.norm(v_T_S, axis=1)

# Energía específica de la órbita Tierra-Sol (por unidad de masa reducida)
mu_TS = MU_SOL + MU_TIERRA
eps_T = 0.5 * v_T_S_mag**2 - mu_TS / d_T_S  # energía específica

# Semieje mayor a = -μ/(2ε)
a_T = -mu_TS / (2 * eps_T)

print(f'  Semieje mayor a (promedio): {a_T.mean():.5f} UA  (esperado: ~1.000 UA)')
print(f'  Variación relativa de a: {(a_T.max()-a_T.min())/a_T.mean():.2e}')

# Vis-viva predicha vs. velocidad obtenida de la integración
v_visviva = np.sqrt(np.abs(mu_TS * (2.0/d_T_S - 1.0/a_T.mean())))  # UA/d
v_visviva_km = v_visviva * AU_m / day_s / 1e3
v_actual_km  = v_T_S_mag  * AU_m / day_s / 1e3

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ts, v_actual_km,  color=COLORES['Tierra'], linewidth=1.5, label='Velocidad integrada')
ax.plot(ts, v_visviva_km, color='k', linewidth=1.0, linestyle='--',
        alpha=0.6, label='Ley vis-viva (con a = cte)')
ax.set_xlabel('Tiempo (días desde 11 abr 2026)')
ax.set_ylabel('Velocidad Tierra–Sol (km/s)')
ax.set_title('Verificación de la Ley Vis-viva para la Tierra')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Error máximo relativo entre vis-viva y simulación: '
      f'{np.max(np.abs(v_actual_km - v_visviva_km)/v_actual_km):.2e}')


---
## 11. Elementos Orbitales y Análisis Adicional

### 11.1 Semieje mayor y excentricidad de Apophis

A partir de la energía y el momento angular específicos de Apophis respecto al Sol, podemos calcular sus elementos orbitales:

- **Semieje mayor:** $a = -\mu_\odot / (2\varepsilon)$ donde $\varepsilon = \frac{v^2}{2} - \frac{\mu_\odot}{r}$
- **Excentricidad:** $e = \sqrt{1 + 2\varepsilon h^2/\mu_\odot^2}$ donde $\mathbf{h} = \mathbf{r} \times \mathbf{v}$


In [ ]:
# ─── Elementos orbitales de Apophis ──────────────────────────────────────
# Posición y velocidad de Apophis relativo al Sol
r_A_S = r_apophis - r_sol   # UA
v_A_S = v_apophis - v_sol   # UA/d

d_A_S   = np.linalg.norm(r_A_S, axis=1)  # distancia Apophis-Sol
v_A_S_m = np.linalg.norm(v_A_S, axis=1)

# Energía específica por unidad de masa de Apophis
mu_AS = MU_SOL  # la masa de Apophis es despreciable
eps_A = 0.5 * v_A_S_m**2 - mu_AS / d_A_S

# Semieje mayor
a_A = -mu_AS / (2 * eps_A)

# Momento angular específico
h_A = np.cross(r_A_S, v_A_S)  # (N,3)
h_A_mag = np.linalg.norm(h_A, axis=1)

# Excentricidad
e_A = np.sqrt(np.abs(1 + 2*eps_A*h_A_mag**2 / mu_AS**2))

print('Elementos orbitales de Apophis respecto al Sol:')
print(f'  Semieje mayor a: {a_A.mean():.5f} UA  (DE441/JPL: ~0.9227 UA)')
print(f'  Excentricidad e: {e_A.mean():.5f}     (DE441/JPL: ~0.1914)')
print(f'  Período orbital: {2*np.pi*np.sqrt(a_A.mean()**3/mu_AS):.2f} días')
print(f'              = {2*np.pi*np.sqrt(a_A.mean()**3/mu_AS)/365.25:.3f} años')
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(ts, a_A, color=COLORES['Apophis'], linewidth=1.5)
ax.axhline(a_A.mean(), color='k', linestyle='--', alpha=0.5, label=f'Media={a_A.mean():.5f} UA')
ax.axhline(0.9227, color='gray', linestyle=':', alpha=0.6, label='JPL DE441: 0.9227 UA')
ax.set_xlabel('Tiempo (días)')
ax.set_ylabel('Semieje mayor a (UA)')
ax.set_title('Semieje mayor de Apophis')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(ts, e_A, color='purple', linewidth=1.5)
ax.axhline(e_A.mean(), color='k', linestyle='--', alpha=0.5, label=f'Media={e_A.mean():.5f}')
ax.axhline(0.1914, color='gray', linestyle=':', alpha=0.6, label='JPL DE441: 0.1914')
ax.set_xlabel('Tiempo (días)')
ax.set_ylabel('Excentricidad e')
ax.set_title('Excentricidad de Apophis')
ax.legend(fontsize=9)

plt.suptitle('Variación de los elementos orbitales de Apophis durante 1 año', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('Las variaciones en a y e son debidas a las perturbaciones gravitacionales de')
print('la Tierra, la Luna y el Sol. En este modelo de 4 cuerpos son relativamente pequeñas.')


---
## 12. Resumen de Resultados


In [ ]:
# ─── Tabla resumen final ─────────────────────────────────────────────────
print('='*80)
print('RESUMEN DE LA SIMULACIÓN')
print(f'Sistema: Sol – Tierra – Luna – Apophis')
print(f'Época inicial: {EPOCH}')
print(f'Duración: 365 días (1 año)  |  Pasos: 1 día  |  N_pasos = {N_PASOS}')
print(f'Sistema de referencia: SSB eclíptico J2000')
print('='*80)

print('\nParámetros gravitacionales (μ = GM):')
for nombre, _, _, mu in CUERPOS:
    print(f'  {nombre:<10}: μ = {mu:.4e} UA³/d²')

print('\nDistancias promedio al SSB:')
for nombre, d in [('Sol', d_sol), ('Tierra', d_tierra), ('Luna', d_luna), ('Apophis', d_apophis)]:
    print(f'  {nombre:<10}: {d.mean():.4f} UA  (min={d.min():.4f}, max={d.max():.4f})')

print('\nVelocidades promedio respecto al SSB:')
for nombre, v_mag in mag_v.items():
    print(f'  {nombre:<10}: {v_mag.mean():.3f} km/s')

print(f'\nError máximo en conservación de energía: {eps_E.max():.2e}')
print(f'Error máximo en conservación de |L|:    {eps_L.max():.2e}')

print('='*80)


---
## 13. Conclusiones

En esta clase aprendimos a:

1. **Obtener vectores de estado** (posición $\mathbf{r}$ y velocidad $\mathbf{v}$) de Sol, Tierra, Luna y Apophis respecto al SSB usando `pymcel` y JPL Horizons, con la función `pc.consulta_horizons(id, location='@0', epochs, datos='vectors')`. El uso del SSB como origen garantiza un marco de referencia inercial donde el momento lineal total se anula.

2. **Caracterizar las masas** de cada cuerpo mediante el parámetro gravitacional $\mu_i = GM_i$, que es medido con mayor precisión que $G$ o $M$ por separado. En unidades UA–día, $\mu_\odot \approx 2.96 \times 10^{-4}$ UA³/d².

3. **Formular el Problema de N-Cuerpos** con sus 4 cuerpos, donde Apophis tiene una masa despreciable ($\mu_\text{Apophis}/\mu_\odot \approx 10^{-20}$) pero su trayectoria sí es afectada por las perturbaciones gravitacionales de los demás cuerpos.

4. **Integrar las ecuaciones de movimiento** durante 1 año con pasos de 1 día usando `pc.ncuerpos_solucion()`, verificando la conservación de la energía y el momento angular.

5. **Visualizar** las trayectorias en 3D y en las proyecciones 2D, analizar las distancias Tierra–Luna y Tierra–Apophis, las velocidades, y verificar la Ley Vis-viva.

### Extensiones posibles

- Incluir **Júpiter** como quinto cuerpo para analizar su efecto sobre Apophis.
- Extender la simulación hasta el **encuentro cercano de abril de 2029** (uso del integrador IAS15 con REBOUND para mayor precisión).
- Agregar la **presión de radiación solar** (efecto Yarkovsky) como aceleración no gravitacional.
- Realizar un análisis de **Monte Carlo** para evaluar la sensibilidad de la trayectoria a las incertidumbres en las condiciones iniciales.

---

### Referencias

1. Zuluaga, J. I. (2024). *Mecánica Celeste: teoría, algoritmos y problemas*. Universidad de Antioquia. DOI: 10.5281/zenodo.18849743
2. Repositorio `pymcel`: https://github.com/seap-udea/pymcel
3. Repositorio `meccel-clase-profesor`: https://github.com/seap-udea/meccel-clase-profesor
4. Park, R. S. et al. (2021). *The Astronomical Journal*, 161, 105. [Efemérides DE441]
5. Giorgini, J. D. et al. (2008). *Icarus*, 193, 1–19. [Apophis en 2029]
6. Rein, H. & Spiegel, D. S. (2015). *MNRAS*, 446, 1424–1437. [Integrador IAS15]
7. IAU Working Group on Numerical Standards (2015). [Constantes astronómicas]
8. JPL Horizons Web Interface: https://ssd.jpl.nasa.gov/horizons/

---
*Cuaderno generado para el curso de Mecánica Celeste – Universidad de Antioquia*  
*Basado en `pymcel` v0.9.x y JPL Horizons DE441*
